# Compute the K-Distance Graph for the SciBERT 10% Stratified Sample
for the k-values 5, 10, 50, 100  
Embeddings for the calculation from SciBERT  
last layer, second to last layer and average of the last four layers

In [1]:
# packages
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [2]:
SAMPLE_PATH = Path("/kaggle/input/datasets/lianestrauch/scibert-samples")

In [3]:
# create function: get_kdistances(file, identifier)

def get_kdistance(
    sample_file: str,
    identifier: str,
    k_values: list[int],
): #-> dict[int, float]:
    """ 
    Calculate k-nearest-neighbor distances for a sample and export the results. 
    
    The function reads embeddings from a Parquet file, 
    computes the distance to the k-th nearest neighbor for each sample 
    using cosine distance, sorts the resulting distances, 
    and saves the results to a CSV file for each value of ``k``. 
    
    Parameters
    ---------- 
    sample_file : str 
        Path to the input Parquet file containing the sample data. 
        The last column is assumed to contain the embedding vectors, 
        and an ``id`` column is required to identify the samples. 
        
    identifier : str 
        Identifier used as part of the output CSV filenames. 
    
    k_values : list[int] 
        List of k values for which k-nearest-neighbor distances should be calculated. 
    
    Raises 
    ------ 
    FileNotFoundError 
        If ``sample_file`` does not exist. 
    
    ValueError 
        If ``sample_file`` does not have a ``.parquet`` extension. 
        
    
    Returns 
    ------- 
    None The function writes one CSV file per k value and does not return a value. 
    
    Notes 
    ----- 
    The output files are named ``k_distance_{identifier}_{k}.csv`` 
    and contain the following columns: 
        - ``rank``: Rank of the sample after sorting by k-distance. 
        - ``id``: mentions_id 
        - ``k_distance``: Distance to the k-th nearest neighbor. 
        The embeddings are compared using cosine distance 
        via :class:`sklearn.neighbors.NearestNeighbors`. 
    """

    if not sample_file.is_file():
        raise FileNotFoundError(
            f"Sample file does not exist: {sample_file}"
        )

    if sample_file.suffix.lower() != ".parquet":
        raise ValueError(
            f"Expected a parquet file, got: {sample_file}"
        )

    sample = pd.read_parquet(sample_file)

    col_name = sample.columns[-1]

    X = np.vstack(sample[col_name].values)

    for k in k_values:
        neighbors = NearestNeighbors(
            n_neighbors=k,
            metric="cosine"
        )
    
        neighbors.fit(X)
    
        distances, _ = neighbors.kneighbors(X)
    
        k_distances = np.sort(distances[:, k - 1])[::-1]
    
        # sort distances and ids together
        sorted_indices = np.argsort(k_distances)
        sorted_distances = k_distances[sorted_indices]
        sorted_ids = sample["id"].values[sorted_indices]

        # Rank after sorting
        ranks = np.arange(len(sorted_distances))

        # Create export DataFrame
        k_distance_df = pd.DataFrame({
        "rank": ranks,
        "id": sorted_ids,
        "k_distance": sorted_distances
        })

        csv_path = f"k_distance_{identifier}_{k}.csv"
        
        k_distance_df.to_csv(
        csv_path,
        index=False
        )
        
        print(f"  Saved: {csv_path}")

    return

#get_kdistance(file = "sample_bert_base_last.parquet", identifier = "bb_ll", k_values = [50, 100])
    
    

In [4]:
all_samples = sorted(SAMPLE_PATH.glob("*.parquet"))
all_samples

[PosixPath('/kaggle/input/datasets/lianestrauch/scibert-samples/sample_scibert_base_2nd_last.parquet'),
 PosixPath('/kaggle/input/datasets/lianestrauch/scibert-samples/sample_scibert_base_last.parquet'),
 PosixPath('/kaggle/input/datasets/lianestrauch/scibert-samples/sample_scibert_base_mean_last4.parquet')]

In [5]:
all_identifier = [str(sample)[72:] for sample in all_samples]
all_identifier = [sample[:-8] for sample in all_identifier]
all_identifier

['2nd_last', 'last', 'mean_last4']

In [6]:
for file, identifier in zip(all_samples, all_identifier):
    get_kdistance(sample_file = file,
             identifier = f"sb_{identifier}", #scibert = sb
             k_values = [5, 10, 50, 100] #on full: 50, 100, 500, 1000
            )

  Saved: k_distance_sb_2nd_last_5.csv
  Saved: k_distance_sb_2nd_last_10.csv
  Saved: k_distance_sb_2nd_last_50.csv
  Saved: k_distance_sb_2nd_last_100.csv
  Saved: k_distance_sb_last_5.csv
  Saved: k_distance_sb_last_10.csv
  Saved: k_distance_sb_last_50.csv
  Saved: k_distance_sb_last_100.csv
  Saved: k_distance_sb_mean_last4_5.csv
  Saved: k_distance_sb_mean_last4_10.csv
  Saved: k_distance_sb_mean_last4_50.csv
  Saved: k_distance_sb_mean_last4_100.csv
